# Fine-tune Wav2Vec2 (CTC) on a South African language — local RTX 3060

Tuned for a single local GPU (RTX 3060, 12GB). If you have the 6GB laptop
variant, halve `per_device_train_batch_size` and raise `gradient_accumulation_steps`
to match.

**Expected data layout**: two CSVs, `train.csv` and `eval.csv`, each with columns:
- `audio_path` — path to a 16kHz mono `.wav` file
- `sentence` — the transcript

Point these at the output of your extraction/normalization pipeline.

## 1. Install dependencies

In [1]:
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa torchcodec

## 2. Check GPU

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

CUDA available: True
GPU: NVIDIA GeForce RTX 3060
VRAM (GB): 12.9


## 3. Config — edit these

In [3]:
LANGUAGE = "tsn"  # tsn=Setswana, nso=Sepedi, ven=Tshivenda
BASE_MODEL = "facebook/wav2vec2-large-xlsr-53"
OUTPUT_DIR = f"./wav2vec2-{LANGUAGE}"
TRAIN_CSV = "train.csv"
EVAL_CSV = "eval.csv"

# RTX 3060 12GB: batch 4-8 with grad accumulation is a safe start.
# If you hit CUDA OOM, drop per_device_train_batch_size to 2-4.
PER_DEVICE_TRAIN_BATCH = 4
GRAD_ACCUM_STEPS = 4
PER_DEVICE_EVAL_BATCH = 4

CHARS_TO_IGNORE = r'[,\?\.!\-\;\:"“%‘”�0-9\[\]\'\_]'

## 4. Imports

In [4]:
import re
import json
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union
from datasets import load_dataset, Audio, Dataset
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
)
import evaluate

## 5. Load and normalize data

In [5]:
dataset_dict = load_dataset(
    "dsfsi-anv/za-african-next-voices-compressed",
    LANGUAGE,
    streaming=True
)

dataset_dict["train"] = Dataset.from_list(list(dataset_dict["train"].take(20)))
dataset_dict["dev_test"] = Dataset.from_list(list(dataset_dict["dev_test"].take(20)))

dataset_dict = dataset_dict.cast_column("audio", Audio(sampling_rate=16000))



In [6]:
def normalize_text(batch):
    if batch["transcript"] is None:
        return batch

    text = batch["transcript"]

    # Remove annotation tags like [pause], [cs], [?], [noise] etc.
    text = re.sub(r'\[.*?\]', '', text)

    # Lowercase everything
    text = text.lower()

    # Keep: a-z, apostrophe, whitespace, and the specific diacritic
    # characters confirmed to exist in this corpus:
    # ê ñ ô ŝ š ȇ ȏ ḓ
    # Everything else (digits, punctuation like ! " ? _ , - etc.)
    # becomes a space rather than being deleted, to avoid gluing words together
    text = re.sub(r"[^a-z'êñôŝšȇȏḓ\s]", " ", text)

    # Collapse repeated whitespace and trim ends
    text = re.sub(r'\s+', ' ', text).strip()

    batch["transcript"] = text
    return batch

In [7]:
def is_valid_transcript(batch):
    t = batch["transcript"]
    return t is not None and isinstance(t, str) and t.strip() != ""

In [8]:
# for split in ["train", "dev_test"]:
#     dataset_dict[split] = dataset_dict[split].filter(
#         lambda x: x["transcript"] is not None and x["transcript"].strip() != "",
#         num_proc=4
#     )


In [9]:
# diacritics = set()
# for split in ["train", "dev_test"]:
#     for t in dataset_dict[split]["transcript"]:
#         if t:
#             diacritics.update(c for c in t.lower() if not re.match(r"[a-z'\s0-9,\.\-\[\]]", c))

# print(sorted(diacritics))

In [10]:
for split in ["train", "dev_test"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=4)
    dataset_dict[split] = dataset_dict[split].map(normalize_text)

Filter (num_proc=4):   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

## 6. Build vocabulary from your transcripts

In [11]:
def extract_chars(batch):
    all_text = " ".join(batch["transcript"])
    return {"vocab": [list(set(all_text))]}

vocab_set = set()
for split in ["train", "dev_test"]:
    v = dataset_dict[split].map(
        extract_chars, batched=True, batch_size=-1,
        keep_in_memory=True, remove_columns=dataset_dict[split].column_names,
    )
    for row in v["vocab"]:
        vocab_set.update(row)

vocab_dict = {v: k for k, v in enumerate(sorted(vocab_set))}
vocab_dict["|"] = vocab_dict.pop(" ", len(vocab_dict))
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

print(f"Vocab size: {len(vocab_dict)}")
vocab_dict

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Vocab size: 30


{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'r': 17,
 's': 18,
 't': 19,
 'u': 20,
 'w': 21,
 'y': 22,
 'ê': 23,
 'ô': 24,
 'š': 25,
 'ȇ': 26,
 'ȏ': 27,
 '|': 0,
 '[UNK]': 28,
 '[PAD]': 29}

## 7. Build processor

In [12]:
tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|"
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000, padding_value=0.0,
    do_normalize=True, return_attention_mask=True,
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

## 8. Preprocess audio + labels

This reads every wav file — the slowest step on a local machine. `num_proc>1` can speed it up if your CPU has spare cores.

In [13]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])

    # replaces the old `with processor.as_target_processor():` context manager
    batch["labels"] = processor.tokenizer(batch["transcript"]).input_ids

    return batch



In [14]:
test_sentence = dataset_dict["train"][0]["transcript"]
test_sentence

'lenaane la tshegetso le tlhabololo ya balemirui'

In [15]:
for split in ["train", "dev_test"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=4)
    dataset_dict[split] = dataset_dict[split].map(
        prepare_dataset,
        remove_columns=dataset_dict[split].column_names,
        num_proc=2,  # bump to e.g. 4 if you have CPU cores to spare
    )


Filter (num_proc=4):   0%|          | 0/20 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/20 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/20 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/20 [00:00<?, ? examples/s]

In [16]:
encoded = processor(text=test_sentence).input_ids
decoded = processor.decode(encoded)

print(f"Original: {test_sentence}")
print(f"Decoded:  {decoded}")

Original: lenaane la tshegetso le tlhabololo ya balemirui
Decoded:  lenane la tshegetso le tlhabololo ya balemirui


In [17]:
def length_ok(batch):
    # rough CTC feasibility check: downsampled frames must exceed label length
    approx_frames = batch["input_length"] // 320
    return approx_frames > len(batch["labels"])

# for split in ["train", "dev_test"]:
#     dataset_dict[split] = dataset_dict[split].filter(length_ok,num_proc=2)

## 9. Data collator

In [18]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

## 10. Metrics (WER / CER)

In [40]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    return {
        "wer": wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": cer_metric.compute(predictions=pred_str, references=label_str),
        "examples": {"prediction": pred_str[:3], "label": label_str[:3]}
    }

## 11. Load model

In [41]:
model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,   # <-- add this
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)
model.freeze_feature_encoder()  # keep pretrained low-level audio features frozen
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-xlsr-53
Key                          | Status     | 
-----------------------------+------------+-
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [42]:
print(model.config.ctc_zero_infinity)

True


In [43]:
print(model.config.ctc_loss_reduction)

mean


In [44]:
# model.config.ctc_zero_infinity = True

## 12. Training arguments

`fp16=True` roughly halves VRAM use on the 3060 — keep it on.

In [45]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    save_strategy="no",   # disables checkpoint saving entirely, turn on for prod
    load_best_model_at_end=False,  # disables loading best model at end, turn on for prod
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    eval_steps=10,
    save_steps=600,
    logging_steps=10,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    num_train_epochs=200,
    fp16=False,
    max_grad_norm=1.0,
    gradient_checkpointing=True,  # trade speed for VRAM headroom,
    save_total_limit=2,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
    report_to=[] #["tensorboard"],
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset_dict["train"],  # for testing, remove .select() to train on full dataset
    eval_dataset=dataset_dict["dev_test"],  # for testing, remove .select() to evaluate on full dataset
    processing_class=processor.feature_extractor,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 13. Train

On a 3060 expect roughly a few hours for a modest low-resource dataset (a few hours of audio). Watch `nvidia-smi` in a terminal alongside this if you want to confirm you're not close to OOM.

In [46]:
trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer,Examples
10,60.505359,14.885254,0.995134,0.904322,"{'prediction': ['ececeuwe', 'eceueueuececeueueueceuecececeue', 'eceueceupe'], 'label': ['lenaane la tshegetso le tlhabololo ya balemirui', 'leno le thusitse batho ba ba jaaka wayne mansfield wa kwa paarl yo a neng a rekisetsa malomagwe kwa marakeng wa motsekapa fa dikolo di tswaletse mo nakong ya fa a ne a santse a tsena sekolo mme ga jaanong o romela diratsuru tsa gagwe kwa dinageng tsa kwa ntle tseo a di lemileng mo polaseng eo ba mo hiriseditseng yona', 'dingwaga di le tse di fetileng fa a ne a le dingwaga di le mme a setse a itshimoletse']}"
20,59.893750,14.812083,0.992701,0.910636,"{'prediction': ['eceueu ee', 'eueueueueueceueueueuececececeue', 'eceueueceue'], 'label': ['lenaane la tshegetso le tlhabololo ya balemirui', 'leno le thusitse batho ba ba jaaka wayne mansfield wa kwa paarl yo a neng a rekisetsa malomagwe kwa marakeng wa motsekapa fa dikolo di tswaletse mo nakong ya fa a ne a santse a tsena sekolo mme ga jaanong o romela diratsuru tsa gagwe kwa dinageng tsa kwa ntle tseo a di lemileng mo polaseng eo ba mo hiriseditseng yona', 'dingwaga di le tse di fetileng fa a ne a le dingwaga di le mme a setse a itshimoletse']}"
30,56.383704,14.681015,1.000000,0.911122,"{'prediction': ['eueueee', 'eueeueueceueueueueeueeueueceueu', 'ecueueueceee'], 'label': ['lenaane la tshegetso le tlhabololo ya balemirui', 'leno le thusitse batho ba ba jaaka wayne mansfield wa kwa paarl yo a neng a rekisetsa malomagwe kwa marakeng wa motsekapa fa dikolo di tswaletse mo nakong ya fa a ne a santse a tsena sekolo mme ga jaanong o romela diratsuru tsa gagwe kwa dinageng tsa kwa ntle tseo a di lemileng mo polaseng eo ba mo hiriseditseng yona', 'dingwaga di le tse di fetileng fa a ne a le dingwaga di le mme a setse a itshimoletse']}"
40,57.334412,14.414001,1.000000,0.898009,"{'prediction': ['eeeeeeee', 'eeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeee', 'eeeeeeeeeeeee'], 'label': ['lenaane la tshegetso le tlhabololo ya balemirui', 'leno le thusitse batho ba ba jaaka wayne mansfield wa kwa paarl yo a neng a rekisetsa malomagwe kwa marakeng wa motsekapa fa dikolo di tswaletse mo nakong ya fa a ne a santse a tsena sekolo mme ga jaanong o romela diratsuru tsa gagwe kwa dinageng tsa kwa ntle tseo a di lemileng mo polaseng eo ba mo hiriseditseng yona', 'dingwaga di le tse di fetileng fa a ne a le dingwaga di le mme a setse a itshimoletse']}"
50,51.444812,10.643417,1.000000,0.950947,"{'prediction': ['', '', ''], 'label': ['lenaane la tshegetso le tlhabololo ya balemirui', 'leno le thusitse batho ba ba jaaka wayne mansfield wa kwa paarl yo a neng a rekisetsa malomagwe kwa marakeng wa motsekapa fa dikolo di tswaletse mo nakong ya fa a ne a santse a tsena sekolo mme ga jaanong o romela diratsuru tsa gagwe kwa dinageng tsa kwa ntle tseo a di lemileng mo polaseng eo ba mo hiriseditseng yona', 'dingwaga di le tse di fetileng fa a ne a le dingwaga di le mme a setse a itshimoletse']}"
60,34.267526,5.688327,1.000000,1.000000,"{'prediction': ['', '', ''], 'label': ['lenaane la tshegetso le tlhabololo ya balemirui', 'leno le thusitse batho ba ba jaaka wayne mansfield wa kwa paarl yo a neng a rekisetsa malomagwe kwa marakeng wa motsekapa fa dikolo di tswaletse mo nakong ya fa a ne a santse a tsena sekolo mme ga jaanong o romela diratsuru tsa gagwe kwa dinageng tsa kwa ntle tseo a di lemileng mo polaseng eo ba mo hiriseditseng yona', 'dingwaga di le tse di fetileng fa a ne a le dingwaga di le mme a setse a itshimoletse']}"
70,22.094827,4.089601,1.000000,1.000000,"{'prediction': ['', '', ''], 'label': ['lenaane la tshegetso le tlhabololo ya balemirui', 'leno le thusitse batho ba ba jaaka wayne mansfield wa kwa paarl yo a neng a rekisetsa malomagwe kwa marakeng wa motsekapa fa dikolo di tswaletse mo nakong ya fa a ne a santse a tsena sekolo mme ga jaanong o romela diratsuru tsa gagwe kwa dinageng tsa kwa ntle tseo a di lemileng mo polaseng eo ba mo hiriseditseng yona', 

TrainOutput(global_step=600, training_loss=14.348516540527344, metrics={'train_runtime': 7514.047, 'train_samples_per_second': 0.532, 'train_steps_per_second': 0.08, 'total_flos': 1.9198265985234335e+18, 'train_loss': 14.348516540527344, 'epoch': 200.0})

In [47]:
trainer.state.log_history

[{'loss': 60.50535888671875,
  'grad_norm': 45.6237678527832,
  'learning_rate': 7.5e-06,
  'epoch': 3.4,
  'step': 10},
 {'eval_loss': 14.88525390625,
  'eval_wer': 0.9951338199513382,
  'eval_cer': 0.9043224866440019,
  'eval_examples': {'prediction': ['ececeuwe',
    'eceueueuececeueueueceuecececeue',
    'eceueceupe'],
   'label': ['lenaane la tshegetso le tlhabololo ya balemirui',
    'leno le thusitse batho ba ba jaaka wayne mansfield wa kwa paarl yo a neng a rekisetsa malomagwe kwa marakeng wa motsekapa fa dikolo di tswaletse mo nakong ya fa a ne a santse a tsena sekolo mme ga jaanong o romela diratsuru tsa gagwe kwa dinageng tsa kwa ntle tseo a di lemileng mo polaseng eo ba mo hiriseditseng yona',
    'dingwaga di le tse di fetileng fa a ne a le dingwaga di le mme a setse a itshimoletse']},
  'eval_runtime': 32.527,
  'eval_samples_per_second': 0.615,
  'eval_steps_per_second': 0.154,
  'epoch': 3.4,
  'step': 10},
 {'loss': 59.89375,
  'grad_norm': 36.14676284790039,
  'learni

## 14. Save

In [48]:
# trainer.save_model(OUTPUT_DIR)
# processor.save_pretrained(OUTPUT_DIR)
# print(f"Saved to {OUTPUT_DIR}")

## 15. Quick sanity-check inference

In [49]:
import torch
import soundfile as sf

In [50]:
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, eleme

In [51]:
sample = dataset_dict["dev_test"].select(range(1))[0]
input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)
print("Raw predicted IDs:", predicted_ids[0].tolist())
print("Unique IDs predicted:", set(predicted_ids[0].tolist()))
print("Pad/blank token ID:", processor.tokenizer.pad_token_id)

Raw predicted IDs: [29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29]
Unique IDs predicted: {29}
Pad/blank token ID: 29


# Test 1: preprocessed dev_test split

In [52]:
def test_on_eval_set(num_samples=5):
    print("=== Evaluation on dev_test split ===\n")
    test_samples = dataset_dict["dev_test"].select(range(num_samples))

    for i, sample in enumerate(test_samples):
        input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(input_values).logits

        predicted_ids = torch.argmax(logits, dim=-1)
        predicted_text = processor.tokenizer.decode(predicted_ids[0])
        actual_text = processor.tokenizer.decode(sample["labels"], group_tokens=False)

        print(f"--- Sample {i+1} ---")
        print(f"Predicted: {predicted_text}")
        print(f"Actual:    {actual_text}\n")

In [53]:
test_on_eval_set(num_samples=5)

=== Evaluation on dev_test split ===

--- Sample 1 ---
Predicted: 
Actual:    lenaane la tshegetso le tlhabololo ya balemirui

--- Sample 2 ---
Predicted: 
Actual:    leno le thusitse batho ba ba jaaka wayne mansfield wa kwa paarl yo a neng a rekisetsa malomagwe kwa marakeng wa motsekapa fa dikolo di tswaletse mo nakong ya fa a ne a santse a tsena sekolo mme ga jaanong o romela diratsuru tsa gagwe kwa dinageng tsa kwa ntle tseo a di lemileng mo polaseng eo ba mo hiriseditseng yona

--- Sample 3 ---
Predicted: 
Actual:    dingwaga di le tse di fetileng fa a ne a le dingwaga di le mme a setse a itshimoletse

--- Sample 4 ---
Predicted: 
Actual:    o ne a gana tšhono ya go dira kwa polaseng eo a neng rwala diratsuru kwa go yona go di rekisa

--- Sample 5 ---
Predicted: 
Actual:    le fa go le jalo dingwaga di se kae fela morago ga seno o ne a fetola mogopolo wa gagwe fa a bona gore diratsuru di rekisiwa thata



# Test 2: raw audio files (informal validation)

In [54]:
def transcribe_audio_file(filepath):
    audio, sr = sf.read(filepath)
    inputs = processor(audio, sampling_rate=sr, return_tensors="pt").input_values.to(device)

    with torch.no_grad():
        logits = model(inputs).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    return processor.tokenizer.decode(predicted_ids[0])

In [55]:
def test_on_raw_files(filepaths):
    print("=== Transcription on raw audio files ===\n")
    for path in filepaths:
        prediction = transcribe_audio_file(path)
        print(f"File: {path}")
        print(f"Predicted: {prediction}\n")

In [56]:
raw_files = [
    "/home/khotso/data/validation_clips/clip1.wav",
    "/home/khotso/data/validation_clips/clip2.wav",
]
# test_on_raw_files(raw_files)

In [57]:
# import soundfile as sf

# test_path = dataset["validation"][0] if False else None  # replace with a real wav path
# # example:
# # speech, sr = sf.read("some_eval_clip.wav")
# # inputs = processor(speech, sampling_rate=16000, return_tensors="pt").input_values.to(model.device)
# # with torch.no_grad():
# #     logits = model(inputs).logits
# # pred_ids = torch.argmax(logits, dim=-1)
# # print(processor.batch_decode(pred_ids))